## 7.3 Plantear las hipótesis

Para este análisis inferencial, utilizaremos la variable continua **Hora del Incidente** (transformada a formato decimal). Para las comparaciones por categorías, seleccionamos dos zonas con dinámicas de movilidad contrastantes: la **Comuna 10 (Centro)**, caracterizada por su alta carga comercial y logística diurna, y la **Comuna 14 (El Poblado)**, caracterizada por el flujo corporativo y su fuerte dinámica de ocio nocturno.

In [1]:
import pandas as pd
import numpy as np
import math
from scipy import stats

# 1. Cargar y preparar datos
df = pd.read_csv('incidentes_viales_motos.csv')

def hora_a_decimal(h):
    try:
        partes = str(h).split(':')
        return int(partes[0]) + int(partes[1])/60 + int(partes[2])/3600
    except: return None

df['HORA_DECIMAL'] = df['HORA_INCIDENTE'].apply(hora_a_decimal)
df = df.dropna(subset=['HORA_DECIMAL', 'ZONA'])

# Extraer grupos
c10 = df[df['ZONA'] == 'COMUNA 10']['HORA_DECIMAL']
c14 = df[df['ZONA'] == 'COMUNA 14']['HORA_DECIMAL']

# Estadísticos base para cálculos manuales
n_tot, mean_tot, std_tot = len(df), df['HORA_DECIMAL'].mean(), df['HORA_DECIMAL'].std()
n10, mean10, var10 = len(c10), c10.mean(), c10.var()
n14, mean14, var14 = len(c14), c14.mean(), c14.var()

print(f"Global: n={n_tot}, Media={mean_tot:.2f}, Desviación={std_tot:.2f}")
print(f"C10 (Centro): n={n10}, Media={mean10:.2f}, Varianza={var10:.2f}")
print(f"C14 (Poblado): n={n14}, Media={mean14:.2f}, Varianza={var14:.2f}")

Global: n=223439, Media=13.53, Desviación=5.52
C10 (Centro): n=34440, Media=13.39, Varianza=26.79
C14 (Poblado): n=20318, Media=13.65, Varianza=26.53


### 1. Prueba de normalidad para la variable continua

* **$H_0$:** La variable "Hora del Incidente" proviene de una distribución normal.
* **$H_1$:** La variable "Hora del Incidente" NO proviene de una distribución normal.

**Análisis manual:** Debido al tamaño masivo de la muestra ($n > 220,000$), la comprobación gráfica (ver numeral 7.2) evidencia una distribución multimodal con picos en las horas pico de tráfico, alejándose de la campana de Gauss tradicional.

**Interpretación en el contexto del caso:** Si el P-valor es menor a 0.05, rechazamos $H_0$. El comportamiento de la accidentalidad en Medellín obedece a las rutinas ciudadanas (desplazamientos de trabajo y estudio), por lo que no es un fenómeno aleatorio y simétrico. Sin embargo, por el Teorema del Límite Central (muestras muy grandes), es posible aplicar pruebas paramétricas para evaluar las medias.

In [2]:
# Prueba de bondad de ajuste de Kolmogorov-Smirnov
stat_ks, p_ks = stats.kstest(df['HORA_DECIMAL'], 'norm', args=(mean_tot, std_tot))
print(f"Estadístico KS: {stat_ks:.4f}")
print(f"P-valor: {p_ks}")

if p_ks < 0.05:
    print("Conclusión: Se rechaza H0. Los datos NO siguen una distribución normal.")
else:
    print("Conclusión: No se rechaza H0. Los datos siguen una distribución normal.")

Estadístico KS: 0.0651
P-valor: 0.0
Conclusión: Se rechaza H0. Los datos NO siguen una distribución normal.


### 2. Prueba de igualdad de varianzas para comparación de categorías
Comparamos las varianzas de la Comuna 10 frente a la Comuna 14.

* **$H_0$:** $\sigma_{10}^2 = \sigma_{14}^2$ (Las varianzas son iguales / Existe homocedasticidad).
* **$H_1$:** $\sigma_{10}^2 \neq \sigma_{14}^2$ (Las varianzas son diferentes).

**Cálculo (Estadístico F de Fisher):**
$$F_{calc} = \frac{S_{10}^2}{S_{14}^2} = \frac{26.79}{26.52} = 1.010$$

**Interpretación en el contexto del caso:** Si confirmamos la igualdad de varianzas, significa que la dispersión de los accidentes a lo largo del día es equivalente en ambas comunas. Es decir, la "ventana de riesgo" dura lo mismo tanto en el Centro como en El Poblado, validando que la variabilidad del tráfico es una constante estructural en las zonas céntricas de la ciudad.

In [3]:
# Prueba F para igualdad de varianzas
f_stat = var10 / var14
df1, df2 = n10 - 1, n14 - 1
p_val_f = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))

print(f"Estadístico F calculado: {f_stat:.3f}")
print(f"P-valor: {p_val_f:.3f}")

if p_val_f < 0.05:
    print("Conclusión: Se rechaza H0. Las varianzas son diferentes.")
else:
    print("Conclusión: No se rechaza H0. Se asume igualdad de varianzas (Homocedasticidad).")

Estadístico F calculado: 1.010
P-valor: 0.422
Conclusión: No se rechaza H0. Se asume igualdad de varianzas (Homocedasticidad).


### 3. Prueba de igualdad de medias para comparación de categorías

* **$H_0$:** $\mu_{10} = \mu_{14}$ (El promedio de hora de los incidentes es igual).
* **$H_1$:** $\mu_{10} \neq \mu_{14}$ (El promedio de hora es diferente).

**Cálculo manual (Estadístico Z para varianzas iguales y muestras grandes):**
$$Z = \frac{\bar{X}_{10} - \bar{X}_{14}}{\sqrt{\frac{S_{p}^2}{n_{10}} + \frac{S_{p}^2}{n_{14}}}} \approx \frac{13.38 - 13.64}{\sqrt{\frac{26.79}{34440} + \frac{26.52}{20318}}}$$

$$Z = \frac{-0.26}{0.045} = -5.77$$

**Interpretación en el contexto del caso:** Dado que el valor absoluto de Z es muy grande, se espera rechazar $H_0$. Los accidentes en El Poblado ocurren estadísticamente más tarde que en el Centro. El Centro tiene su dinámica fuerte desde la madrugada (plazas de mercado, abastecimiento) y decae al finalizar la tarde. En contraste, El Poblado activa zonas gastronómicas y de entretenimiento nocturno, arrastrando la media de accidentalidad hacia horas más tardías.

In [4]:
# Prueba T para diferencia de medias (asumiendo varianzas iguales según prueba anterior)
t_stat, p_t = stats.ttest_ind(c10, c14, equal_var=True)

print(f"Estadístico T calculado: {t_stat:.3f}")
print(f"P-valor: {p_t}")

if p_t < 0.05:
    print("Conclusión: Se rechaza H0. Existe una diferencia significativa entre las medias.")
else:
    print("Conclusión: No se rechaza H0. Las medias son estadísticamente iguales.")

Estadístico T calculado: -5.751
P-valor: 8.93179212177276e-09
Conclusión: Se rechaza H0. Existe una diferencia significativa entre las medias.


## 7.4 Intervalos de confianza

### 1. Intervalo al 95% para la variable completa (Hora promedio en Medellín)

**Cálculo:**
$$IC = \bar{X} \pm Z_{\alpha/2} \left( \frac{S}{\sqrt{n_{tot}}} \right) = 13.52 \pm 1.96 \left( \frac{5.52}{\sqrt{223439}} \right)$$
$$IC = 13.52 \pm 0.022 \implies [13.49 , 13.54]$$

**Interpretación en el contexto del caso:** Con un nivel de confianza del 95%, el promedio real de la hora en la que ocurren los incidentes de motocicletas en toda Medellín se sitúa entre las 13.49 (1:29 PM) y las 13.54 (1:32 PM). El margen de error es mínimo debido al volumen masivo de registros históricos, lo que otorga a las autoridades de tránsito un rango de tiempo exacto y confiable para enfocar campañas de prevención justo después del mediodía.

In [5]:
# IC Global en software
z_val = 1.96
margen_global = z_val * (std_tot / math.sqrt(n_tot))
ic_inf_glob = mean_tot - margen_global
ic_sup_glob = mean_tot + margen_global

print(f"IC 95% Global: [{ic_inf_glob:.2f} , {ic_sup_glob:.2f}] horas.")

IC 95% Global: [13.50 , 13.55] horas.


### 2. Intervalos de confianza para cada comparación de categorías (Diferencia de Medias)

**Cálculo (Centro vs Poblado):**
$$IC = (\bar{X}_{10} - \bar{X}_{14}) \pm Z_{\alpha/2} \sqrt{\frac{S_{10}^2}{n_{10}} + \frac{S_{14}^2}{n_{14}}}$$
$$IC = (13.38 - 13.64) \pm 1.96 (0.045)$$
$$IC = -0.26 \pm 0.088 \implies [-0.34 , -0.17]$$

**Interpretación en el contexto del caso:** Con un 95% de confianza, la diferencia entre las medias es estrictamente negativa (no pasa por el cero), lo que confirma que en el Centro los incidentes ocurren, en promedio, entre 10 minutos (0.17 h) y 20 minutos (0.34 h) más temprano que en El Poblado. Esto proporciona a la Secretaría de Movilidad un sustento cuantitativo para escalar o diferir los horarios operativos de los agentes de tránsito dependiendo de la comuna a intervenir.

In [6]:
# IC Diferencia de Medias en software
diferencia_medias = mean10 - mean14
error_estandar_dif = math.sqrt((var10/n10) + (var14/n14))
margen_dif = z_val * error_estandar_dif

ic_inf_dif = diferencia_medias - margen_dif
ic_sup_dif = diferencia_medias + margen_dif

print(f"Diferencia de medias (Centro - Poblado): {diferencia_medias:.2f} horas.")
print(f"IC 95% Diferencia: [{ic_inf_dif:.2f} , {ic_sup_dif:.2f}] horas.")

Diferencia de medias (Centro - Poblado): -0.26 horas.
IC 95% Diferencia: [-0.35 , -0.17] horas.
